In [1]:
import pandas as pd
import numpy as np
import spacy
from spacy.tokens import Doc
from spacytextblob.spacytextblob import SpacyTextBlob
from pathlib import Path

In [2]:
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("spacytextblob")

In [3]:
CORPUS_PATH = Path("corpus")
GENDER_PATHS = [p for p in CORPUS_PATH.iterdir() if p.is_dir()]
GENDER_PATHS

[WindowsPath('corpus/equal'),
 WindowsPath('corpus/genderless'),
 WindowsPath('corpus/man'),
 WindowsPath('corpus/woman')]

In [4]:
def get_paragraphs_df(gender_path: Path):
    files = [p for p in gender_path.iterdir() if p.is_file()]

    def loop():
        for file in files:
            with open(file, encoding="utf-8") as f:
                yield f.readlines()
    
    paragraphs_per_file = list(loop())
    
    df = pd.DataFrame({
        "file": files,
        "gender": gender_path.stem,
        "paragraph": paragraphs_per_file,
    })

    df = df \
        .explode(column="paragraph") \
        .replace("\n", np.nan) \
        .dropna()
    
    df["paragraph_index"] = [i + 1 for i in range(df.shape[0])]

    return df

In [7]:
full_df = pd.concat(get_paragraphs_df(p) for p in GENDER_PATHS)

# Map gender categories to numeric values
gender_mapping = {
    gender: i + 1 
    for i, gender in enumerate(sorted(full_df["gender"].unique()))
}
full_df["gender_index"] = full_df["gender"].map(gender_mapping)
full_df = full_df.set_index(keys=["gender_index", "paragraph_index"])

full_df.head()

file  \
gender_index paragraph_index                                                      
1            1                corpus\equal\news_benoit-saint-denis-the-god-o...   
             2                corpus\equal\news_chicago-prelim-results-du-pl...   
             3                corpus\equal\news_cody-brundage-needed-reset-u...   
             4                corpus\equal\news_fight-by-fight-preview-ufc-3...   
             5                corpus\equal\news_fight-by-fight-preview-ufc-3...   

                             gender  \
gender_index paragraph_index          
1            1                equal   
             2                equal   
             3                equal   
             4                equal   
             5                equal   

                                                                      paragraph  
gender_index paragraph_index                                                     
1            1                Although there are still mountains to climb be...  
             2                When the scorecards were collected and totalle...  
             3                “When I took the time off (after the Alhassan ...  
             4                In the main event, undisputed UFC heavyweight ...  
             5                Main Event: Tom Aspinall vs Ciryl GaneCo-Main ...

In [8]:
# This cell might take a while, have patience
full_df["doc"] = list(nlp.pipe(full_df["paragraph"]))
full_df.head()

file  \
gender_index paragraph_index                                                      
1            1                corpus\equal\news_benoit-saint-denis-the-god-o...   
             2                corpus\equal\news_chicago-prelim-results-du-pl...   
             3                corpus\equal\news_cody-brundage-needed-reset-u...   
             4                corpus\equal\news_fight-by-fight-preview-ufc-3...   
             5                corpus\equal\news_fight-by-fight-preview-ufc-3...   

                             gender  \
gender_index paragraph_index          
1            1                equal   
             2                equal   
             3                equal   
             4                equal   
             5                equal   

                                                                      paragraph  \
gender_index paragraph_index                                                      
1            1                Although there are still mountains to climb be...   
             2                When the scorecards were collected and totalle...   
             3                “When I took the time off (after the Alhassan ...   
             4                In the main event, undisputed UFC heavyweight ...   
             5                Main Event: Tom Aspinall vs Ciryl GaneCo-Main ...   

                                                                            doc  
gender_index paragraph_index                                                     
1            1                (Although, there, are, still, mountains, to, c...  
             2                (When, the, scorecards, were, collected, and, ...  
             3                (“, When, I, took, the, time, off, (, after, t...  
             4                (In, the, main, event, ,, undisputed, UFC, hea...  
             5                (Main, Event, :, Tom, Aspinall, vs, Ciryl, Gan...

In [9]:
def count_instances(doc: Doc, lemma: str | list[str]):
    if not isinstance(lemma, list):
        lemma = [lemma]

    return sum(
        1
        for token in doc
        if token.lemma_ in lemma
    )

In [10]:
# Nouns/Pronouns
full_df["n_champion"] = full_df["doc"].apply(count_instances, args=("champion",))
full_df["n_win/wins"] = full_df["doc"].apply(count_instances, args=(["win", "wins"],))
full_df["n_loss/losses"] = full_df["doc"].apply(count_instances, args=(["loss", "losses"],))
full_df["n_fight"] = full_df["doc"].apply(count_instances, args=("fight",))
full_df["n_unanimous"] = full_df["doc"].apply(count_instances, args=("unanimous",))
full_df["n_split"] = full_df["doc"].apply(count_instances, args=("split",))
full_df["n_decision"] = full_df["doc"].apply(count_instances, args=("decision",))
full_df["n_performance"] = full_df["doc"].apply(count_instances, args=("performance",))
full_df["n_knockout"] = full_df["doc"].apply(count_instances, args=("knockout",))
full_df["n_submission"] = full_df["doc"].apply(count_instances, args=("submission",))
full_df["n_woman/women"] = full_df["doc"].apply(count_instances, args=(["woman", "women"],))
full_df["n_man/men"] = full_df["doc"].apply(count_instances, args=(["man", "men"],))
full_df["n_she/her/hers/herself"] = full_df["doc"].apply(count_instances, args=(["she", "her", "hers", "herself"],))
full_df["n_he/him/his/himself"] = full_df["doc"].apply(count_instances, args=(["n_he", "him", "his", "himself"],))

full_df.head()

file  \
gender_index paragraph_index                                                      
1            1                corpus\equal\news_benoit-saint-denis-the-god-o...   
             2                corpus\equal\news_chicago-prelim-results-du-pl...   
             3                corpus\equal\news_cody-brundage-needed-reset-u...   
             4                corpus\equal\news_fight-by-fight-preview-ufc-3...   
             5                corpus\equal\news_fight-by-fight-preview-ufc-3...   

                             gender  \
gender_index paragraph_index          
1            1                equal   
             2                equal   
             3                equal   
             4                equal   
             5                equal   

                                                                      paragraph  \
gender_index paragraph_index                                                      
1            1                Although there are still mountains to climb be...   
             2                When the scorecards were collected and totalle...   
             3                “When I took the time off (after the Alhassan ...   
             4                In the main event, undisputed UFC heavyweight ...   
             5                Main Event: Tom Aspinall vs Ciryl GaneCo-Main ...   

                                                                            doc  \
gender_index paragraph_index                                                      
1            1                (Although, there, are, still, mountains, to, c...   
             2                (When, the, scorecards, were, collected, and, ...   
             3                (“, When, I, took, the, time, off, (, after, t...   
             4                (In, the, main, event, ,, undisputed, UFC, hea...   
             5                (Main, Event, :, Tom, Aspinall, vs, Ciryl, Gan...   

                              n_champion  n_win/wins  n_loss/losses  n_fight  \
gender_index paragraph_index                                                   
1            1                         0           0              0        1   
             2                         0           2              0        0   
             3                         0           0              0        8   
             4                         1           0              0        1   
             5                         0           0              0        0   

                              n_unanimous  n_split  n_decision  n_performance  \
gender_index paragraph_index                                                    
1            1                          0        0           0              0   
             2                          0        0           0              0   
             3                          0        0           0              0   
             4                          0        0           0              0   
             5                          0        0           0              0   

                              n_knockout  n_submission  n_woman/women  \
gender_index paragraph_index                                            
1            1                         0             0              0   
             2                         0             0              0   
             3                         0             0              0   
             4                         0             0              0   
             5                         0             0              0   

                              n_man/men  n_she/her/hers/herself  \
gender_index paragraph_index                                      
1            1                        0                       0   
             2                        0                       1   
             3                        1                       1   
             4                        0                       0   
            

In [11]:
# Adjectives
full_df["a_good"] = full_df["doc"].apply(count_instances, args=("good",))
full_df["a_great"] = full_df["doc"].apply(count_instances, args=("great",))
full_df["a_top"] = full_df["doc"].apply(count_instances, args=("top",))
full_df["a_big"] = full_df["doc"].apply(count_instances, args=("big",))
full_df["a_competitive"] = full_df["doc"].apply(count_instances, args=("competitive",))
full_df["a_dominant"] = full_df["doc"].apply(count_instances, args=("dominant",))
full_df["a_dangerous"] = full_df["doc"].apply(count_instances, args=("dangerous",))
full_df["a_impressive"] = full_df["doc"].apply(count_instances, args=("impressive",))
full_df["a_interesting"] = full_df["doc"].apply(count_instances, args=("interesting",))
full_df["a_elite"] = full_df["doc"].apply(count_instances, args=("elite",))
full_df["a_aggressive"] = full_df["doc"].apply(count_instances, args=("aggressive",))
full_df["a_outstanding"] = full_df["doc"].apply(count_instances, args=("outstanding",))
full_df["a_entertaining"] = full_df["doc"].apply(count_instances, args=("entertaining",))
full_df["a_exciting"] = full_df["doc"].apply(count_instances, args=("exciting",))
full_df["a_real"] = full_df["doc"].apply(count_instances, args=("real",))
full_df["a_tough"] = full_df["doc"].apply(count_instances, args=("tough",))
full_df["a_unbeaten"] = full_df["doc"].apply(count_instances, args=("unbeaten",))
full_df["a_difficult"] = full_df["doc"].apply(count_instances, args=("difficult",))

full_df.head()

file  \
gender_index paragraph_index                                                      
1            1                corpus\equal\news_benoit-saint-denis-the-god-o...   
             2                corpus\equal\news_chicago-prelim-results-du-pl...   
             3                corpus\equal\news_cody-brundage-needed-reset-u...   
             4                corpus\equal\news_fight-by-fight-preview-ufc-3...   
             5                corpus\equal\news_fight-by-fight-preview-ufc-3...   

                             gender  \
gender_index paragraph_index          
1            1                equal   
             2                equal   
             3                equal   
             4                equal   
             5                equal   

                                                                      paragraph  \
gender_index paragraph_index                                                      
1            1                Although there are still mountains to climb be...   
             2                When the scorecards were collected and totalle...   
             3                “When I took the time off (after the Alhassan ...   
             4                In the main event, undisputed UFC heavyweight ...   
             5                Main Event: Tom Aspinall vs Ciryl GaneCo-Main ...   

                                                                            doc  \
gender_index paragraph_index                                                      
1            1                (Although, there, are, still, mountains, to, c...   
             2                (When, the, scorecards, were, collected, and, ...   
             3                (“, When, I, took, the, time, off, (, after, t...   
             4                (In, the, main, event, ,, undisputed, UFC, hea...   
             5                (Main, Event, :, Tom, Aspinall, vs, Ciryl, Gan...   

                              n_champion  n_win/wins  n_loss/losses  n_fight  \
gender_index paragraph_index                                                   
1            1                         0           0              0        1   
             2                         0           2              0        0   
             3                         0           0              0        8   
             4                         1           0              0        1   
             5                         0           0              0        0   

                              n_unanimous  n_split  ...  a_interesting  \
gender_index paragraph_index                        ...                  
1            1                          0        0  ...              0   
             2                          0        0  ...              0   
             3                          0        0  ...              0   
             4                          0        0  ...              0   
             5                          0        0  ...              0   

                              a_elite  a_aggressive  a_outstanding  \
gender_index paragraph_index                                         
1            1                      0             0              0   
             2                      0             0              0   
             3                      0             0              0   
             4                      0             0              0   
             5                      0             0              0   

                              a_entertaining  a_exciting  a_real  a_tough  \
gender_index paragraph_index                                                
1            1                             0           0       0        0   
             2                             0           0       0        0   
             3                             0           0       0        0   
             4                             0           0       0        0   
             5        

In [12]:
full_df["paragraph_length"] = full_df["doc"].apply(lambda doc: len(doc))
full_df.head()

file  \
gender_index paragraph_index                                                      
1            1                corpus\equal\news_benoit-saint-denis-the-god-o...   
             2                corpus\equal\news_chicago-prelim-results-du-pl...   
             3                corpus\equal\news_cody-brundage-needed-reset-u...   
             4                corpus\equal\news_fight-by-fight-preview-ufc-3...   
             5                corpus\equal\news_fight-by-fight-preview-ufc-3...   

                             gender  \
gender_index paragraph_index          
1            1                equal   
             2                equal   
             3                equal   
             4                equal   
             5                equal   

                                                                      paragraph  \
gender_index paragraph_index                                                      
1            1                Although there are still mountains to climb be...   
             2                When the scorecards were collected and totalle...   
             3                “When I took the time off (after the Alhassan ...   
             4                In the main event, undisputed UFC heavyweight ...   
             5                Main Event: Tom Aspinall vs Ciryl GaneCo-Main ...   

                                                                            doc  \
gender_index paragraph_index                                                      
1            1                (Although, there, are, still, mountains, to, c...   
             2                (When, the, scorecards, were, collected, and, ...   
             3                (“, When, I, took, the, time, off, (, after, t...   
             4                (In, the, main, event, ,, undisputed, UFC, hea...   
             5                (Main, Event, :, Tom, Aspinall, vs, Ciryl, Gan...   

                              n_champion  n_win/wins  n_loss/losses  n_fight  \
gender_index paragraph_index                                                   
1            1                         0           0              0        1   
             2                         0           2              0        0   
             3                         0           0              0        8   
             4                         1           0              0        1   
             5                         0           0              0        0   

                              n_unanimous  n_split  ...  a_elite  \
gender_index paragraph_index                        ...            
1            1                          0        0  ...        0   
             2                          0        0  ...        0   
             3                          0        0  ...        0   
             4                          0        0  ...        0   
             5                          0        0  ...        0   

                              a_aggressive  a_outstanding  a_entertaining  \
gender_index paragraph_index                                                
1            1                           0              0               0   
             2                           0              0               0   
             3                           0              0               0   
             4                           0              0               0   
             5                           0              0               0   

                              a_exciting  a_real  a_tough  a_unbeaten  \
gender_index paragraph_index                                            
1            1                         0       0        0           0   
             2                         0       0        0           0   
             3                         0       0        0           0   
             4                         0       0        0           0   
             5                         

In [13]:
full_df["polarity"] = full_df["doc"].apply(lambda doc: doc._.blob.polarity)
full_df["polarity_percent"] = (full_df["polarity"] * 100).round(0).astype(int)
full_df.head()

file  \
gender_index paragraph_index                                                      
1            1                corpus\equal\news_benoit-saint-denis-the-god-o...   
             2                corpus\equal\news_chicago-prelim-results-du-pl...   
             3                corpus\equal\news_cody-brundage-needed-reset-u...   
             4                corpus\equal\news_fight-by-fight-preview-ufc-3...   
             5                corpus\equal\news_fight-by-fight-preview-ufc-3...   

                             gender  \
gender_index paragraph_index          
1            1                equal   
             2                equal   
             3                equal   
             4                equal   
             5                equal   

                                                                      paragraph  \
gender_index paragraph_index                                                      
1            1                Although there are still mountains to climb be...   
             2                When the scorecards were collected and totalle...   
             3                “When I took the time off (after the Alhassan ...   
             4                In the main event, undisputed UFC heavyweight ...   
             5                Main Event: Tom Aspinall vs Ciryl GaneCo-Main ...   

                                                                            doc  \
gender_index paragraph_index                                                      
1            1                (Although, there, are, still, mountains, to, c...   
             2                (When, the, scorecards, were, collected, and, ...   
             3                (“, When, I, took, the, time, off, (, after, t...   
             4                (In, the, main, event, ,, undisputed, UFC, hea...   
             5                (Main, Event, :, Tom, Aspinall, vs, Ciryl, Gan...   

                              n_champion  n_win/wins  n_loss/losses  n_fight  \
gender_index paragraph_index                                                   
1            1                         0           0              0        1   
             2                         0           2              0        0   
             3                         0           0              0        8   
             4                         1           0              0        1   
             5                         0           0              0        0   

                              n_unanimous  n_split  ...  a_outstanding  \
gender_index paragraph_index                        ...                  
1            1                          0        0  ...              0   
             2                          0        0  ...              0   
             3                          0        0  ...              0   
             4                          0        0  ...              0   
             5                          0        0  ...              0   

                              a_entertaining  a_exciting  a_real  a_tough  \
gender_index paragraph_index                                                
1            1                             0           0       0        0   
             2                             0           0       0        0   
             3                             0           0       0        0   
             4                             0           0       0        0   
             5                             0           0       0        0   

                              a_unbeaten  a_difficult  paragraph_length  \
gender_index paragraph_index                                              
1            1                         0            0                33   
             2                         0            0                68   
             3                         0            0               298   
             4                         0            0       

In [15]:
COLS_TO_DELETE = [
    "file",
    "gender",
    "paragraph",
    "doc",
]

In [16]:
df_to_save = full_df.drop(COLS_TO_DELETE, axis=1)
df_to_save.head()

n_champion  n_win/wins  n_loss/losses  n_fight  \
gender_index paragraph_index                                                   
1            1                         0           0              0        1   
             2                         0           2              0        0   
             3                         0           0              0        8   
             4                         1           0              0        1   
             5                         0           0              0        0   

                              n_unanimous  n_split  n_decision  n_performance  \
gender_index paragraph_index                                                    
1            1                          0        0           0              0   
             2                          0        0           0              0   
             3                          0        0           0              0   
             4                          0        0           0              0   
             5                          0        0           0              0   

                              n_knockout  n_submission  ...  a_outstanding  \
gender_index paragraph_index                            ...                  
1            1                         0             0  ...              0   
             2                         0             0  ...              0   
             3                         0             0  ...              0   
             4                         0             0  ...              0   
             5                         0             0  ...              0   

                              a_entertaining  a_exciting  a_real  a_tough  \
gender_index paragraph_index                                                
1            1                             0           0       0        0   
             2                             0           0       0        0   
             3                             0           0       0        0   
             4                             0           0       0        0   
             5                             0           0       0        0   

                              a_unbeaten  a_difficult  paragraph_length  \
gender_index paragraph_index                                              
1            1                         0            0                33   
             2                         0            0                68   
             3                         0            0               298   
             4                         0            0               108   
             5                         0            0                17   

                              polarity  polarity_percent  
gender_index paragraph_index                              
1            1                0.000000                 0  
             2                0.457143                46  
             3                0.222650                22  
             4                0.202381                20  
             5                0.166667                17  

[5 rows x 35 columns]

In [17]:
OUTPUT_FILE = "paragraph_stats.csv"
df_to_save.to_csv(OUTPUT_FILE)